# 34 — Reproducible Research Pipelines, Experiment Tracking, and Packaging

Notebooks are excellent for learning and exploration, but serious research projects need stable code, explicit configs, repeatable commands, and structured outputs.

We will study:

- Moving from notebooks to a project
- Configuration files
- Reusable modules
- Command-line training
- Experiment directories
- Logging
- Checkpoint naming
- Environment capture
- Testing
- Data/model versioning
- Packaging
- Reproducible result generation


In [ ]:
import json
import platform
import sys
import hashlib
from pathlib import Path
import torch

print("Python:",sys.version)
print("PyTorch:",torch.__version__)


# 1. Why Move Beyond Notebooks?

Notebooks can hide execution state. A script starts from a clean process and exposes missing dependencies.


# 2. Recommended Project Structure

```text
project/
├── configs/
├── data/
├── metadata/
├── splits/
├── src/
│   ├── datasets.py
│   ├── transforms.py
│   ├── models.py
│   ├── train.py
│   ├── evaluate.py
│   └── utils.py
├── experiments/
├── tests/
├── requirements.txt
└── README.md
```


# 3. Configuration Files

A config makes experiment choices explicit rather than hidden in code.


In [ ]:
config={
    "experiment_name":"baseline_resnet18",
    "seed":42,
    "image_size":224,
    "batch_size":32,
    "learning_rate":1e-4,
    "weight_decay":1e-4,
    "epochs":20,
    "model":"resnet18",
    "primary_metric":"macro_f1",
    "split_manifest":"splits/patient_split.json"
}
print(json.dumps(config,indent=2))


# 4. Save Config Before Training


In [ ]:
Path("example_config.json").write_text(
    json.dumps(config,indent=2),
    encoding="utf-8"
)


# 5. Config as an Experiment Contract

Avoid hidden defaults. If a choice matters to the result, record it.


# 6. Reusable Modules

Stable logic belongs in functions/classes that can be imported by training and evaluation scripts.


In [ ]:
def build_model_from_config(config):
    if config["model"]=="tiny_mlp":
        return torch.nn.Sequential(
            torch.nn.Linear(16,32),
            torch.nn.ReLU(),
            torch.nn.Linear(32,3)
        )
    raise ValueError(f"Unknown model: {config['model']}")


# 7. Command-Line Training

A project can expose a reproducible command such as:

```bash
python -m src.train --config configs/exp001.json
```


# 8. Experiment Directory

Every run should have its own output directory.


In [ ]:
def experiment_dir(root,name,seed):
    p=Path(root)/f"{name}_seed{seed}"
    p.mkdir(parents=True,exist_ok=True)
    return p

run_dir=experiment_dir("experiments","baseline",42)
print(run_dir)


# 9. What to Save

Per run:

- Config
- History
- Best checkpoint
- Last checkpoint
- Predictions
- Metrics
- Environment information


# 10. Stable Checkpoint Naming

Prefer:

```text
best_val_loss.pth
best_macro_f1.pth
last_epoch.pth
```

over ambiguous manual names.


In [ ]:
def checkpoint_dict(model,optimizer,epoch,best_metric,config):
    return {
        "model_state_dict":model.state_dict(),
        "optimizer_state_dict":optimizer.state_dict(),
        "epoch":epoch,
        "best_metric":best_metric,
        "config":config
    }


# 11. Logging

At minimum record:

- Epoch
- Train loss
- Validation loss
- Primary metric
- Learning rate


In [ ]:
history=[]
def log_epoch(epoch,train_loss,val_loss,val_metric,lr):
    history.append({
        "epoch":epoch,
        "train_loss":train_loss,
        "val_loss":val_loss,
        "val_metric":val_metric,
        "learning_rate":lr
    })


# 12. Experiment Tracking Tools

Possible tools:

- TensorBoard
- Weights & Biases
- MLflow
- Plain CSV/JSON

The key requirement is reproducible metadata.


# 13. Environment Capture


In [ ]:
environment={
    "python":sys.version,
    "platform":platform.platform(),
    "torch":torch.__version__,
    "cuda_available":torch.cuda.is_available(),
    "cuda_version":torch.version.cuda
}
print(json.dumps(environment,indent=2))


# 14. Split Versioning

The exact patient split is part of the experiment.

Save IDs, not only the seed.


# 15. Data Versioning

If files or metadata change, old experiments may no longer be reproducible.

Use immutable snapshots, versions, or hashes.


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()


# 16. Testing

Research code benefits from tests for:

- Dataset shapes
- No patient overlap
- Model output shape
- Deterministic preprocessing
- Metric correctness


In [ ]:
def test_model_shape():
    model=torch.nn.Linear(8,3)
    x=torch.randn(5,8)
    assert model(x).shape==(5,3)

test_model_shape()
print("Shape test passed.")


# 17. Leakage Test


In [ ]:
def assert_disjoint_groups(train_ids,val_ids,test_ids):
    assert set(train_ids).isdisjoint(val_ids)
    assert set(train_ids).isdisjoint(test_ids)
    assert set(val_ids).isdisjoint(test_ids)


# 18. Smoke Tests

Run a tiny end-to-end training job after major code changes:

- Few samples
- One epoch
- One checkpoint

The goal is execution correctness, not performance.


# 19. Continuous Integration Intuition

Automated tests can run after every code change or pull request.


# 20. Separate Configuration From Code

Hyperparameters should come from config instead of being hidden deep inside functions.


# 21. Separate Data Paths From Model Logic

Model classes should not know where CSV files live.


# 22. Reusable Evaluation Script

A clean evaluator should accept:

- Checkpoint
- Config
- Split
- Output path

and regenerate predictions consistently.


# 23. Save Raw Predictions

Prediction-level outputs allow later recalculation of metrics, calibration, bootstrap intervals, and error analyses without rerunning inference.


# 24. Reproducible Result Tables

Generate final tables programmatically from stored result files.

Avoid manual copy/paste.


# 25. Reproducible Figures

Plots should be generated from saved data using code.


# 26. Run Registry


In [ ]:
run_registry=[]

def register_run(run_id,config_path,result_path):
    run_registry.append({
        "run_id":run_id,
        "config_path":str(config_path),
        "result_path":str(result_path)
    })


# 27. Packaging

For larger projects:

```bash
pip install -e .
```

Then imports become stable:

```python
from project.models import build_model
```


# 28. `pyproject.toml`

Modern projects commonly use `pyproject.toml` for package metadata and dependencies.


# 29. Notebook Role After Packaging

Notebooks remain excellent for:

- Exploration
- Visualization
- Error analysis
- Teaching

But core training/evaluation logic should live in reusable modules.


# 30. README

A research README should explain:

- Research question
- Data schema
- Installation
- Training
- Evaluation
- Reproduction steps


# 31. Reproduction Goal

Ideally:

```bash
python -m src.train --config configs/baseline.json
python -m src.evaluate --run experiments/baseline_seed42
```

should reproduce the result.


# 32. Common Mistakes

- Notebook-only hidden state
- Overwriting old runs
- No exact split manifest
- Missing preprocessing config
- No software versions
- Hand-built final result tables


# 33. Exercises

1. Create a JSON config.
2. Build a model factory.
3. Create experiment directories.
4. Save a checkpoint dictionary.
5. Capture environment metadata.
6. Hash a metadata file.
7. Write a patient-overlap test.
8. Write a model-shape test.
9. Design a project tree.
10. Draft reproducibility instructions.


# 34. Key Takeaways

A reproducible result requires:

$$
\boxed{
Config+Code+Split+Checkpoint+Predictions+Environment
}
$$

The goal is that another researcher—or future you—can rerun the experiment without guessing.


# Next Notebook

# 35 — Final Capstone: A Research-Quality Ultrasound Deep Learning Project

In the final notebook, we will combine the entire course into one end-to-end research workflow.
